# 🌰 Mazorca IA — Google Colab (GRATIS)
Asistente de cacao con **visión + web + diálogo**. Corre gratis con GPU.

**Pasos:** Entorno de ejecución → *Cambiar tipo de entorno* → **GPU T4**. Luego ejecuta las celdas en orden.
Para el diálogo (opcional), agrega tu token HF en Secretos de Colab con nombre `HF_TOKEN`.

In [ ]:
# 1. Instalar dependencias (Gradio fijado para el chat)
!pip install -q gradio==4.44.1 ultralytics huggingface_hub ddgs opencv-python-headless

In [ ]:
# 2. (Opcional) token HF para el dialogo LLM libre
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass  # sin token: funciona igual con vision + web

In [ ]:
# 3. Lanzar Mazorca IA (da un link publico share)

import os, numpy as np, cv2, gradio as gr
from huggingface_hub import hf_hub_download

MODEL_PATH = hf_hub_download("juanvilla/cacao-modelo", "best.pt")
LLM_MODEL = os.environ.get("MAZORCA_LLM", "meta-llama/Llama-3.2-3B-Instruct")
HF_TOKEN  = os.environ.get("HF_TOKEN")
REMAP = {"Fitoftora": "Phytophthora", "Monilia": "Moniliophthora"}
SISTEMA = ("Eres 'Mazorca IA', asistente experto y cercano de cacao (Theobroma cacao) "
           "para el Valle del Cauca. Ayudas a estudiantes y agricultores en espanol claro "
           "y practico. Usa el diagnostico de vision y la busqueda web si te los dan. No inventes.")
_yolo = None
def yolo():
    global _yolo
    if _yolo is None:
        from ultralytics import YOLO; _yolo = YOLO(MODEL_PATH)
    return _yolo
def diagnosticar(img):
    if img is None: return None
    m = yolo(); r = m.predict(cv2.cvtColor(img, cv2.COLOR_RGB2BGR), conf=0.25, verbose=False)[0]
    if r.boxes is None or len(r.boxes) == 0: return "No detecte cacao claro en la foto."
    c = {}
    for cl, cf in zip(r.boxes.cls.cpu().numpy().astype(int), r.boxes.conf.cpu().numpy()):
        n = REMAP.get(m.names[cl], m.names[cl]); c[n] = max(c.get(n, 0), float(cf))
    return "Diagnostico visual: " + ", ".join(f"{k} ({v:.0%})" for k, v in c.items()) + "."
def buscar_web(q, n=3):
    try:
        from ddgs import DDGS
        with DDGS() as d: res = list(d.text(f"{q} cacao Theobroma", max_results=n))
        return "\n".join(f"- {r.get('title','')}: {r.get('body','')[:220]}" for r in res)
    except Exception: return ""
def llm(msgs):
    if not HF_TOKEN: return None
    try:
        from huggingface_hub import InferenceClient
        return InferenceClient(model=LLM_MODEL, token=HF_TOKEN).chat_completion(messages=msgs, max_tokens=500, temperature=0.4).choices[0].message.content
    except Exception as e: return f"(LLM no disponible: {str(e)[:80]})"
def responder(msg, img, hist):
    diag = diagnosticar(img) if img is not None else None
    web = buscar_web(msg) if msg and len(msg) > 3 else ""
    ctx = (f"\n[Vision] {diag}" if diag else "") + (f"\n[Web]\n{web}" if web else "")
    msgs = [{"role": "system", "content": SISTEMA}]
    for u, a in hist[-4:]: msgs += [{"role": "user", "content": u}, {"role": "assistant", "content": a}]
    msgs.append({"role": "user", "content": (msg or "Analiza la foto.") + ctx})
    r = llm(msgs)
    if r is None:
        p = [x for x in [diag, ("Web:\n"+web if web else "")] if x]
        return "\n\n".join(p) if p else "Sube una foto o pregunta sobre cacao. (El dialogo se activa con HF_TOKEN.)"
    return r
with gr.Blocks(title="Mazorca IA") as demo:
    gr.Markdown("# 🌰 Mazorca IA\nAsistente libre de cacao (Valle del Cauca): vision + web + dialogo.")
    chat = gr.Chatbot(height=380)
    with gr.Row():
        img = gr.Image(label="Foto (opcional)", type="numpy", height=140)
        with gr.Column(scale=3):
            msg = gr.Textbox(label="Mensaje", placeholder="Que le pasa a mi mazorca?")
            btn = gr.Button("Enviar", variant="primary")
    def send(m, i, h):
        return h + [[m or "(foto)", responder(m, i, h)]], None, None
    btn.click(send, [msg, img, chat], [chat, msg, img]); msg.submit(send, [msg, img, chat], [chat, msg, img])
demo.launch(share=True)
